# Forecasting Hourly Day-Ahead Electricity Prices in the German-Luxembourg Bidding Zone
**DAI Mission — Data & AI in Economics | TU Dortmund**

---

## Team

| Name | Role |
|------|------|
| Lennart Oberkönig | Lead / All Components |
| Tim Janis Schmale | All Components |

---

**LLM Assistance Disclosure - TODO:**  
*[Required if applicable] We used [tool name] to [brief description, e.g., debug DoWhy syntax].
All analysis, interpretation, and conclusions are our own.*

## Research Question

*How accurately can hourly day-ahead electricity prices in the German-Luxembourg bidding zone be forecasted using market fundamentals, renewable generation forecasts, load forecasts and calendar effects?*

In [68]:
## Packages

# basic packages
import numpy as np
import pandas as pd
from pathlib import Path
import re

# plotting
import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import seaborn as sns
 

# sklearn models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.compose import TransformedTargetRegressor

# preprocessing and pipelines
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# metrics and model inspection
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# clustering and manifold learning
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

# others
import holidays
from itertools import combinations

# Causal Inference - TODO
from dowhy import CausalModel
import networkx as nx

np.random.seed(42)
print('All imports successful')

All imports successful


---
## Work Plan - TODO

| Section | Responsible member | Main tasks |
|---------|-------------------|------------|
| §1 Research Question & Data | Lennart Oberkönig & Tim Schmale | data sourcing, cleaning, variable table |
| §2 Causal Inference | Lennart Oberkönig & Tim Schmale | DAG design, DoWhy implementation, refutation |
| §3 Supervised Learning | Lennart Oberkönig & Tim Schmale | model selection, training, evaluation |
| §4 Unsupervised / Generative | Lennart Oberkönig & Tim Schmale | method choice, implementation, visualisation |
| §5 Synthesis & Communication | Lennart Oberkönig & Tim Schmale | cross-method narrative, conclusion, notebook readability |

**Shared tasks:** The whole project was done together in person.

---
## Section 1 — Research Question & Data

### Research Question

How accurately can hourly day-ahead electricity prices in the German-Luxembourg bidding zone be forecasted using market fundamentals, renewable generation forecasts, load forecasts and calendar effects?

### Motivation

Electricity price forecasting is a challenging and highly relevant task in modern power markets because short-term electricity prices exhibit complex dynamics and depend on the continuous balance between production and consumption, which is affected by several factors such as demand and weather conditions (Maciejowska, Uniejewski and Weron, 2022, P. 1ff.). In day-ahead electricity markets, market participants submit buy and sell orders for electricity delivery on the following day. These bids and offers are aggregated into demand and supply curves, and the market-clearing price is determined by the intersection of these curves. Thus, the hourly day-ahead price reflects the equilibrium between expected electricity demand and available supply for each delivery hour. The auction for this pricing mechanism closes each day at 12:00 and the prices for the next day are determined (Ghelasi and Ziel, 2024, P. 588f.).

Beyond its methodological relevance, electricity price forecasting also has direct economic value. Accurate day-ahead price forecasts can support market participants in planning bidding strategies, scheduling generation or consumption, managing price risk and identifying economically favorable hours for flexible assets such as storage or demand-side flexibility. In this sense, forecast accuracy is not only a statistical objective but can translate into better market decisions.

The auction-based price formation is closely related to the merit-order effect. Since electricity from renewable energy sources such as wind and solar PV is characterized by negligible marginal costs, increasing renewable feed-in tends to affect the aggregated supply curve and can reduce day-ahead electricity prices (Macedo, Marques and Damette, 2022, P. 885ff.). Therefore, renewable generation is an important explanatory factor for forecasting hourly day-ahead electricity prices. This mechanism is particularly relevant for the German-Luxembourg bidding zone, where electricity prices are closely linked to load and renewable generation. Another driving factor for the price is seasonality, since the price is showing recurring patterns on a weekly, daily and intraday level (Trebbien et al., 2024, P. 35f.). 

From a forecasting perspective, this leads to an important modeling question: whether day-ahead electricity prices should be represented as one continuous hourly time series or as a 24-dimensional daily price vector. In a univariate framework, hourly prices are treated as one high-frequency time series, and forecasts for the 24 hours of the next day are generated sequentially. This means that earlier forecasts can enter the prediction of later hours, which makes the approach sensitive to error accumulation. In contrast, the multivariate framework uses an explicit day-by-hour structure and forecasts all 24 hourly prices of the next day at once. This allows each delivery hour to have its own model structure and to capture hour-specific price patterns (Ziel and Weron, 2018, P. 397ff.). In addition, this framework can be extended by including explanatory variables such as load forecasts, wind and solar generation forecasts and calendar effects. Due to the inclusion of additional explanatory variables we choose a multivariate modeling framework for forecasting the hourly day-ahead prices.

This project combines explanatory and predictive methods to analyze hourly day-ahead electricity prices in the German-Luxembourg bidding zone: a directed acyclic graph is used to structure the assumed relationships between relevant market drivers, K-Means clustering and t-SNE are applied to explore and visualize recurring price regimes, and Decision Tree, Random Forest and Neural Network regression models are evaluated against a naive baseline to assess their forecasting performance.


### Data Sources

| Dataset | Source / URL | Access method |
|---------|-------------|---------------|
| Forecasted Day-Ahead Generation | https://www.smard.de/home/downloadcenter/download-marktdaten/ | local .csv file |
| Forecasted Day-Ahead Load | https://www.smard.de/home/downloadcenter/download-marktdaten/ | local .csv file |
| Day-Ahead Electricity Price | https://www.smard.de/home/downloadcenter/download-marktdaten/ | local .csv file |

In [ ]:
## Load data
# notebook working directory
BASE = Path().resolve()
DATA = BASE / "data"

# load the data
load_forecast = pd.read_csv(DATA / "Prognostizierter_Stromverbrauch.csv",delimiter=";")
generation_forecast = pd.read_csv(DATA / "Prognostizierte_Erzeugung_Day-Ahead.csv",delimiter=";")
day_ahead_price = pd.read_csv(DATA / "Gro_handelspreise.csv",delimiter=";")

# restrict to the desired columns, rename columns & set nans
generation_forecast = generation_forecast[[
    "Datum von",
    "Wind Offshore [MWh] Berechnete Auflösungen",	
    "Wind Onshore [MWh] Berechnete Auflösungen",	
    "Photovoltaik [MWh] Berechnete Auflösungen",	
    "Sonstige [MWh] Berechnete Auflösungen"]].rename(columns={"Datum von": "timestamp",
                                                              "Wind Offshore [MWh] Berechnete Auflösungen": "Wind Offshore Production FC [MWh]",
                                                              "Wind Onshore [MWh] Berechnete Auflösungen": "Wind Onshore Production FC [MWh]",
                                                              "Photovoltaik [MWh] Berechnete Auflösungen": "Photovoltaik Production FC [MWh]",
                                                              "Sonstige [MWh] Berechnete Auflösungen": "Other Production FC [MWh]"})

load_forecast = load_forecast[[
    "Datum von",
    "Netzlast [MWh] Berechnete Auflösungen"]].rename(columns={"Datum von": "timestamp",
                                                              "Netzlast [MWh] Berechnete Auflösungen": "Total Load FC [MWh]",})

day_ahead_price = day_ahead_price[[
    "Datum von", "Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen"]].rename(columns={
    "Datum von": "timestamp",
    "Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen": "Day Ahead Price [EUR/MWh]"})

## Prepare data
# Datetime Objects
generation_forecast["timestamp"] = pd.to_datetime(generation_forecast["timestamp"], dayfirst=True)
load_forecast["timestamp"] = pd.to_datetime(load_forecast["timestamp"], dayfirst=True)
day_ahead_price["timestamp"] = pd.to_datetime(day_ahead_price["timestamp"], dayfirst=True)

## Merge Tables
df_join = pd.merge(pd.merge(load_forecast, generation_forecast, on="timestamp", how="inner"), day_ahead_price,on="timestamp",how="inner")

# Numeric columns are created
df_cleaned = df_join.replace("-", np.nan)
cols = df_cleaned.columns[1:]
for col in cols:
    s = (
        df_cleaned[col]
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )

    # Convert literal "nan" or empty strings to real NaN
    s = s.replace(["nan", ""], pd.NA)

    # Now safely convert to numeric
    df_cleaned[col] = pd.to_numeric(s, errors="coerce")

## first feature engineering
# Holidays
de = holidays.DE()
lu = holidays.LU()
df_cleaned["date only"] = df_cleaned["timestamp"].dt.date
df_cleaned["German_holiday"] = df_cleaned["date only"].apply(lambda x: x in de)
df_cleaned["Luxembourg_holiday"] = df_cleaned["date only"].apply(lambda x: x in lu)
df_cleaned.drop(columns=["date only"], inplace=True)

# Timestamp patterns
df_cleaned["weekday"] = df_cleaned["timestamp"].dt.weekday
df_cleaned["hour"] = df_cleaned["timestamp"].dt.hour
df_cleaned["month"] = df_cleaned["timestamp"].dt.month
df_cleaned["year"] = df_cleaned["timestamp"].dt.year

df_cleaned.head()

,timestamp,Total Load FC [MWh],Wind Offshore Production FC [MWh],Wind Onshore Production FC [MWh],Photovoltaik Production FC [MWh],Other Production FC [MWh],Day Ahead Price [EUR/MWh],German_holiday,Luxembourg_holiday,weekday,hour,month,year
0,2018-10-01 02:00:00,42628.00,1750.50,4152.00,0.0,37311.5,51.41,False,False,0,2,10,2018
1,2018-10-01 03:00:00,42986.75,1895.25,4436.25,0.0,36318.5,47.38,False,False,0,3,10,2018
2,2018-10-01 04:00:00,44675.00,2138.25,4816.25,0.0,37481.5,47.59,False,False,0,4,10,2018
3,2018-10-01 05:00:00,48813.25,2368.50,5276.00,0.0,41073.5,51.61,False,False,0,5,10,2018
4,2018-10-01 06:00:00,57869.00,2649.25,5625.25,0.0,45581.5,69.13,False,False,0,6,10,2018


### Dataset Information

| Variable| Type     | Role (feature / target / instrument / ...)    | Description|
|----------------------------------------------|----------|-----------------------------------------------|----------------------------------------------------------------------------------------|
| timestamp| datetime| Identifier | Start of timeperiod in Central European (Summer-) Time|
| Total Load FC [MWh] | float | feature | Forecasted total electricity consumption for the following day.|
| Wind Offshore Production FC [MWh] | float | feature | Forecasted net electricity generation from offshore wind turbines for the following day.|
| Wind Onshore Production FC [MWh] | float | feature | Forecasted net electricity generation from onshore wind turbines for the following day.|
| Photovoltaik Production FC [MWh] | float | feature | Forecasted net electricity generation from photovoltaic systems for the following day. The forecast is part of the SMARD category for forecasted wind and photovoltaic generation. |
| Other Production FC [MWh] | float | feature | Forecasted net electricity generation from other systems  for the following day. |
| Day Ahead Price [EUR/MWh] | float | target | Hourly wholesale electricity price in the day-ahead market.|
| German_holiday | boolean | feature | Indicator for whether the calendar date of the timestamp is a public holiday in Germany. |
| Luxembourg_holiday | boolean | feature | Indicator for whether the calendar date of the timestamp is a public holiday in Luxembourg. |
| weekday | integer / categorical | feature | Calendar weekday derived from the timestamp. |
| hour | integer / categorical | feature | Hour of the day derived from the timestamp. |
| month | integer / categorical | feature | Calendar month derived from the timestamp. |
| year | integer | feature | Calendar year derived from the timestamp. |

### Data Quality Handling
1. Missing weather data and forecasts
2. Time shifts (missing and doubled hours)
3. Missing Values

These quality issues are handeled in the following.

#### Time Shifts

For the time shift of European time, two central problems appear in our data:
1. Doubled Hours, when time is shifted backwards
    - The doubled hour is averaged out of the data (orientation for problem handling by Ziel & Weron, 2018)
2. Missing Hours, when time is shifted forwards
    - The missing hour is forward filled (orientation for problem handling by Ziel & Weron, 2018)

In [62]:
# average duplicated timestamps
df_time_cleaned = (
    df_cleaned
    .groupby("timestamp", as_index=False)
    .mean(numeric_only=True)
)

# keep original timestamps to detect newly inserted rows later
original_timestamps = df_time_cleaned["timestamp"].copy()

# create a complete hourly time index
full_range = pd.date_range(
    start=df_time_cleaned["timestamp"].min(),
    end=df_time_cleaned["timestamp"].max(),
    freq="h"
)

# reindex to full hourly range (introduces NaNs for missing timestamps)
df_time_cleaned = (
    df_time_cleaned
    .set_index("timestamp")
    .reindex(full_range)
)
df_time_cleaned.index.name = "timestamp"

# identify rows that did not exist before (DST gaps)
new_rows = ~df_time_cleaned.index.isin(original_timestamps)

# forward-fill all values
df_ffill = df_time_cleaned.ffill()

# fill only the newly inserted timestamps with forward-filled values
df_time_cleaned.loc[new_rows, :] = df_ffill.loc[new_rows, :]

# restore timestamp as a column
df_time_cleaned = df_time_cleaned.reset_index()

Take-Away: Missing and double hours due to time shifting are eliminated from the dataset in such a way that all timestamp are completely unique.

#### Missing Values

In [63]:
# check for empty values
df_time_cleaned.isna().sum()

timestamp                               0
Total Load FC [MWh]                  1033
Wind Offshore Production FC [MWh]       0
Wind Onshore Production FC [MWh]        3
Photovoltaik Production FC [MWh]        3
Other Production FC [MWh]            1688
Day Ahead Price [EUR/MWh]               0
German_holiday                          0
Luxembourg_holiday                      0
weekday                                 0
hour                                    0
month                                   0
year                                    0
dtype: int64

To address the missing values in all 4 features, we perform data imputation using the kNN method. This way, the empty values are set to a more realistic value compared to applications of forward fill or other methods, especially when multiple values are missing continuously.

In [64]:
# define the columns which have empty values
impute_targets = df_time_cleaned.columns[df_time_cleaned.isna().any()]

# define the columns used for prediction
predictor_cols = df_time_cleaned.columns[
    ~df_time_cleaned.columns.isin(["timestamp","Day Ahead Price [EUR/MWh]"])
]

# working copy
df = df_time_cleaned.copy()
# cache for fitted models
model_cache = {}

# imputation by kNN
for target in impute_targets:

    # do not include target column in predictors
    candidate_predictors = [col for col in predictor_cols if col != target]

    # get all rows where the target is missing
    missing_indices = df_time_cleaned.index[df_time_cleaned[target].isna()]

    if len(missing_indices) == 0:
        print(f"{target}: no missing values")
        continue

    # iterate over empty rows
    for idx in missing_indices:

        # get all available predictors (non-NaN) for this row
        available_predictors = [
            col for col in candidate_predictors
            if pd.notna(df_time_cleaned.loc[idx, col])
        ]

        # at least one predictor must be available
        if len(available_predictors) == 0:
            continue

        predictors = tuple(available_predictors)

        # training rows: target present + predictors present
        train_mask = (
            df_time_cleaned[target].notna()
            & df_time_cleaned[list(predictors)].notna().all(axis=1)
        )

        X_train = df_time_cleaned.loc[train_mask, list(predictors)]
        y_train = df_time_cleaned.loc[train_mask, target]
        # skip when not enough train data is available
        if len(X_train) < 2:
            continue

        # build model if not cached
        if predictors not in model_cache:
            k = min(5, len(X_train))
            knn = Pipeline([
                ("scaler", StandardScaler()),
                ("knn", KNeighborsRegressor(
                    n_neighbors=k,
                    weights="distance"
                ))
            ])
            knn.fit(X_train, y_train)
            model_cache[predictors] = knn

        # predict with kNN
        X_pred = df_time_cleaned.loc[[idx], list(predictors)]
        prediction = model_cache[predictors].predict(X_pred)[0]


        df.loc[idx, target] = prediction

print(df[impute_targets].isna().sum())

Total Load FC [MWh]                 0
Wind Onshore Production FC [MWh]    0
Photovoltaik Production FC [MWh]    0
Other Production FC [MWh]           0
dtype: int64


Take Away: All missing values are eliminated

### Data Investigation

Prior to the data modeling part, the data is investigated using correlation matrix, time-series plots, and target variable distributions

#### Correlation Matrix

In [74]:
# build correlation matrix
cols = df.columns.drop("timestamp")
corr_matrix = df[cols].corr()

# build the plot
plt.figure(figsize=(16, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"label": "Correlation"}
)
plt.title("Correlation Matrix", fontsize=18, pad=20)
plt.tight_layout()
plt.savefig('plots/corr_matrix.png', dpi=100, bbox_inches='tight')
plt.close()
print('Correlation matrix saved \u2192 plots/corr_matrix.png')

Correlation matrix saved → plots/corr_matrix.png


Take-Away:
- The two wind production forecasts have a strong positive correlation, which is expected given that both are driven by similar meteorological conditions.
- Load and total production are positively correlated, reflecting typical market dynamics: higher demand requires higher generation levels.
- The positive correlation between day‑ahead prices and conventional (non‑renewable) production suggests that prices tend to rise when more traditional generation is required.
- The slightly positive correlation between the year variable and renewable production indicates a structural increase in renewable generation capacity over time.
- This trend is underlined by the negative correlation between the year variable and conventional production.
- The positive correlation between day‑ahead prices and the year variable points to a long‑term upward trend in electricity prices, potentially driven by inflation, geopolitical events, and broader market disruptions such as wars or pandemics.


#### Time-Series Plots

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

# restrict to relevant columns
cols = df.columns.drop("timestamp")[:6]

# build line plots iteratively
for col in cols:
    fig, ax = plt.subplots(figsize=(16, 4))

    ax.plot(df["timestamp"], df[col], color="royalblue", linewidth=2)

    ax.set_title(f"{col} over time", fontsize=16, pad=12)
    ax.set_xlabel("time")
    ax.set_ylabel(col)

    plt.tight_layout()
    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", col).strip("_")
    fname = f"plots/{safe_name}_over_time.png"

    plt.savefig(fname, dpi=100, bbox_inches="tight")
    plt.close()
    print(f'{col} saved \u2192 {fname}')


Total Load FC [MWh] saved → plots/Total_Load_FC_MWh_over_time.png
Wind Offshore Production FC [MWh] saved → plots/Wind_Offshore_Production_FC_MWh_over_time.png
Wind Onshore Production FC [MWh] saved → plots/Wind_Onshore_Production_FC_MWh_over_time.png
Photovoltaik Production FC [MWh] saved → plots/Photovoltaik_Production_FC_MWh_over_time.png
Other Production FC [MWh] saved → plots/Other_Production_FC_MWh_over_time.png
Day Ahead Price [EUR/MWh] saved → plots/Day_Ahead_Price_EUR_MWh_over_time.png


Take-Away:
- The time-series plots indicate expected seasonality patterns for load and production figures.
- The Price time series as underlines the slightly increasing price mean over time.
- Also, the price variance gets larger as the years go on, but do not reach such extremes that methods for variance stabilization will be applied.
- Not needing variance stabilization fits to the conclusions of Ziel and Weron from 2018.

#### Target Variable Distribution

In [ ]:
# create histogram of day ahead price
fig, ax = plt.subplots(figsize=(14, 5))

ax.hist(
    df["Day Ahead Price [EUR/MWh]"],
    bins=100,
    color="steelblue",
    edgecolor="white",
    alpha=0.9
)

ax.set_title("Distribution of Day-Ahead Price [€/MWh]", fontsize=16, pad=12)
ax.set_xlabel("Day-Ahead Price [€/MWh]")
ax.set_ylabel("Count")

ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("plots/day_ahead_price_distribution.png", dpi=100, bbox_inches="tight")
plt.close()
print('Day-Ahead Price distribution saved \u2192 plots/day_ahead_price_distribution.png')


Day-Ahead Price distribution saved → plots/day_ahead_price_distribution.png


In [ ]:
# marker if price is negative or positive
df["negative_price_flag"] = (df["Day Ahead Price [EUR/MWh]"] < 0).astype(int)

# build distribution by hour of day
neg_dist = df.groupby(["hour", "negative_price_flag"]).size().unstack()
neg_dist = neg_dist.rename(columns={
    0: "Positive prices",
    1: "Negative prices"
})


# convert to long format for plotly
neg_dist_plot = neg_dist.reset_index().melt(
    id_vars="hour",
    value_vars=["Positive prices", "Negative prices"],
    var_name="type",
    value_name="count"
)

hours = neg_dist.index
pos = neg_dist["Positive prices"]
neg = neg_dist["Negative prices"]

# stacked bar plot for positive and negative price distribution by hour
plt.figure(figsize=(14, 5))
plt.bar(hours, pos, label="Positive prices", color="steelblue")
plt.bar(hours, neg, bottom=pos, label="Negative prices", color="firebrick")

plt.title("Positive and Negative Prices by Hour", fontsize=16, pad=12)
plt.xlabel("Hour of Day")
plt.ylabel("Number of Observations")
plt.xticks(range(24))
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.legend()

plt.tight_layout()
plt.savefig("plots/negative_price_distribution.png", dpi=100, bbox_inches="tight")
plt.close()

print('Negative Price distribution saved \u2192 plots/negative_price_distribution.png')


# drop column again
df.drop(columns=["negative_price_flag"], inplace=True)

Negative Price distribution saved → plots/negative_price_distribution.png


Take-Away:
- Distribution of prices is rather left skewed on the positive side of 0.
- positive and negative price outliers exist with positive outliers being more frequent than negative outliers.
- Negative prices seem to occur mostly around midday which suits the thesis that renewable energies, especially Photovoltaik are causing this phenomenon

---
## Section 2 — Causal Inference Block

**[TEMPLATE] Rubric checklist (4 pts total):**
- [ ] A causal graph (DAG) is constructed and the assumed relationships are justified
- [ ] Identification strategy is appropriate (backdoor / IV / propensity score)
- [ ] Estimation is implemented correctly using DoWhy (or equivalent)
- [ ] At least one refutation test is run and its result is interpreted

---
*This example uses backdoor adjustment (linear regression). Replace with the strategy appropriate for your research question.*

In [ ]:
# [EXAMPLE — replace with your causal graph]
# Draw the DAG using networkx — no graphviz system package required
G = nx.DiGraph()
G.add_edges_from([
    ('age',       'training'),
    ('education', 'training'),
    ('distance',  'training'),   # instrument: affects treatment but not outcome directly
    ('age',       'log_wage'),
    ('education', 'log_wage'),
    ('training',  'log_wage'),   # causal effect of interest
])

pos = {
    'distance':  (0.0,  0.0),
    'age':       (1.2,  1.2),
    'education': (1.2, -1.2),
    'training':  (2.8,  0.0),
    'log_wage':  (4.4,  0.0),
}
node_colors = [
    '#AED6F1' if n == 'distance' else
    '#FDEBD0' if n in ('age', 'education') else
    '#F9E79F' if n == 'training' else
    '#A9DFBF'   # log_wage
    for n in G.nodes()
]

fig, ax = plt.subplots(figsize=(9, 4))
nx.draw_networkx(G, pos=pos, ax=ax, node_color=node_colors,
                 node_size=2500, font_size=9, arrows=True,
                 arrowsize=20, edge_color='dimgray', width=1.5)
ax.set_title('Causal DAG: Job-Training Programme \u2192 Log Wage', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig('dag.png', dpi=100, bbox_inches='tight')
plt.close()
print('DAG saved \u2192 dag.png')
print('Blue=instrument | Orange=confounders | Yellow=treatment | Green=outcome')

DAG saved → dag.png
Blue=instrument | Orange=confounders | Yellow=treatment | Green=outcome


/var/folders/38/ctwgvy3x5cnc_nvdscq0fbhm0000gn/T/ipykernel_50583/2781548213.py:28: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(9, 4))


In [ ]:
# [EXAMPLE — replace variable names and graph string for your study]
dot_graph = (
    'digraph {'
    ' age -> training; education -> training; distance -> training;'
    ' age -> log_wage; education -> log_wage; training -> log_wage;'
    '}'
)

model = CausalModel(
    data=df,
    treatment='training',
    outcome='log_wage',
    graph=dot_graph,
)

identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print(identified_estimand)

In [ ]:
# [EXAMPLE — backdoor.linear_regression is the simplest estimator; choose what fits your design]
causal_estimate = model.estimate_effect(
    identified_estimand,
    method_name='backdoor.linear_regression',
    target_units='ate',
)
ate = causal_estimate.value
print(causal_estimate)
print(f'\nEstimated Average Treatment Effect (ATE): {ate:.4f}')
print(
    f'Interpretation: training participation is associated with a '
    f'{ate:+.2%} change in log wages, holding age and education constant.'
)

In [ ]:
# [EXAMPLE — random_common_cause refuter]
# Adds a random variable as a common cause; a robust estimate should not change substantially
refutation = model.refute_estimate(
    identified_estimand,
    causal_estimate,
    method_name='random_common_cause',
    num_simulations=5,
)
print(refutation)
print('\n[TEMPLATE] Interpret the refutation: did the estimate remain stable?')
print('If the new estimate is close to the original, the result is robust to this check.')

---
## Section 3 — Supervised Learning Block

**[TEMPLATE] Rubric checklist (4 pts total):**
- [ ] Model choice is justified relative to the prediction task
- [ ] Train/test split and cross-validation are used correctly
- [ ] An appropriate metric is reported and interpreted (RMSE, AUC, accuracy, …)
- [ ] Results are compared to a baseline or alternative model

---
*This example predicts training participation (binary classification) from worker characteristics.
Replace with your own prediction task — classification or regression.*

In [ ]:
# [EXAMPLE — replace X, y, and model choices for your task]
# Prediction task: who participates in training? (binary classification)
X = df[['age', 'education', 'distance']]
y = df['training']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Baseline: logistic regression
lr = LogisticRegression(max_iter=500)
lr.fit(X_train, y_train)
lr_proba = lr.predict_proba(X_test)[:, 1]

# Alternative: random forest
rf = RandomForestClassifier(n_estimators=100, n_jobs=1, random_state=42)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]

# ROC curves
fig, ax = plt.subplots(figsize=(6, 5))
for label, proba in [('Logistic Regression (baseline)', lr_proba),
                     ('Random Forest', rf_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_score   = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{label} (AUC = {auc_score:.3f})', linewidth=1.8)

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves: Predicting Training Participation')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=100, bbox_inches='tight')
plt.close()
print('ROC curves saved \u2192 roc_curves.png')

In [ ]:
# [EXAMPLE — 5-fold cross-validation on the full dataset]
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='roc_auc', n_jobs=1)
print(f'Random Forest | 5-fold CV AUC: {cv_scores.mean():.3f} \u00b1 {cv_scores.std():.3f}')
print(f'Per-fold:      {[round(s, 3) for s in cv_scores]}')

**[TEMPLATE] Interpret your supervised learning results:**

- **Why this model?** *Justify your model choice for your specific prediction task and data type.*
- **Key metric:** *Report and interpret your chosen metric — why is it appropriate for this task?*
- **Baseline comparison:** *How does your best model compare to the simpler baseline?*
- **Limitations:** *Any overfitting concerns? Class imbalance? Feature leakage risks?*

---
## Section 4 — Unsupervised / Generative Block

**[TEMPLATE] Rubric checklist (4 pts total):**
- [ ] Method choice is justified relative to the structure of the data or task
- [ ] Implementation is correct (k selection, linkage choice, latent dim, …)
- [ ] Output is evaluated with an appropriate measure (silhouette, reconstruction loss, …)
- [ ] Findings are visualised and interpreted in domain terms

---
*This example clusters workers into latent types using K-Means + PCA.
Replace with your method: hierarchical clustering, VAE, GAN, topic model, etc.*

In [ ]:
# [EXAMPLE — K-Means on standardised features, visualised in PCA space]
features = df[['age', 'education', 'log_wage']]
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(features)

# PCA for 2-D visualisation
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Elbow curve: choose k
ks       = range(2, 7)
inertias = [KMeans(n_clusters=k, random_state=42, n_init='auto').fit(X_scaled).inertia_ for k in ks]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(ks), inertias, 'o-', color='steelblue')
axes[0].set_xlabel('k (number of clusters)')
axes[0].set_ylabel('Inertia (within-cluster sum of squares)')
axes[0].set_title('Elbow Curve')
axes[0].grid(alpha=0.3)

k_chosen = 3
km       = KMeans(n_clusters=k_chosen, random_state=42, n_init='auto')
labels   = km.fit_predict(X_scaled)
sil      = silhouette_score(X_scaled, labels)
print(f'K={k_chosen} | Silhouette score: {sil:.3f} (range: -1 to +1; higher is better)')

sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', s=10, alpha=0.6)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
axes[1].set_title(f'K-Means Clusters (k={k_chosen}) in PCA Space')
plt.colorbar(sc, ax=axes[1], label='Cluster')
plt.tight_layout()
plt.savefig('clusters.png', dpi=100, bbox_inches='tight')
plt.close()
print('Cluster plot saved \u2192 clusters.png')

In [ ]:
# [EXAMPLE — cluster centroids back in original feature scale]
df['cluster'] = labels
centroids_orig = scaler.inverse_transform(km.cluster_centers_)
centroid_df = pd.DataFrame(
    centroids_orig,
    columns=['age', 'education', 'log_wage'],
    index=[f'Cluster {i}' for i in range(k_chosen)]
)
centroid_df['n_workers']     = df.groupby('cluster').size().values
centroid_df['training_rate'] = df.groupby('cluster')['training'].mean().values
print('Cluster centroids (original scale):')
print(centroid_df.round(2))

**[TEMPLATE] Interpret your unsupervised results:**

- **Why this method?** *Justify in relation to the data structure and research question.*
- **How was k (or another hyperparameter) chosen?** *Describe the trade-off you observed.*
- **What do the clusters mean economically?** *Describe each cluster in plain language: who are these workers?*
- **Limitations:** *Sensitivity to initialisation? Non-spherical clusters? Curse of dimensionality?*

---
## Section 5 — Synthesis & Communication

**[TEMPLATE] Rubric checklist (4 pts total):**
- [ ] The three method blocks are connected — each result informs the next
- [ ] Conclusions directly answer the research question
- [ ] Limitations and potential confounders are honestly discussed
- [ ] Notebook is readable: clear markdown narrative, labelled plots, no dead code

---
*Replace the toy narrative below with your own synthesis.*

### What causal inference revealed

*[TEMPLATE] Summarise your ATE estimate and what it implies for the research question.
Was the effect statistically and economically meaningful?*

**Toy example:** Backdoor adjustment estimates an ATE of approximately +0.40 log-wage units,
suggesting the training programme substantially raises wages after controlling for age and
education. The random-common-cause refuter confirms the estimate is robust to an added
spurious confounder.

---

### What supervised learning revealed

*[TEMPLATE] What does the predictive model tell you about who participates?
Which features matter most? How does this connect to the causal story?*

**Toy example:** Distance to the training centre is the strongest predictor of participation
(as designed — it is the instrument). The Random Forest outperforms logistic regression
(AUC \u2248 0.85 vs \u2248 0.78), suggesting nonlinear selection effects.

---

### What clustering revealed

*[TEMPLATE] How do the clusters relate to your treatment and outcome?
Do certain worker types benefit more from training?*

**Toy example:** Three worker types emerge: (0) young/low-education workers with the highest
training rate; (1) prime-age/highly-educated workers with the highest baseline wage;
(2) older/medium-education workers with the lowest training rate.

---

### Limitations & honest discussion

*[TEMPLATE — required for full marks] Discuss what your analysis cannot establish.
Examples: unmeasured confounders, external validity, distributional assumptions,
model misspecification, data representativeness.*

---

### Conclusion

*[TEMPLATE] 2–3 sentences directly answering the research question stated in Section 1.*

In [ ]:
# [EXAMPLE — 3-panel summary figure connecting all three method blocks]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: ATE point estimate
axes[0].bar(['Training\nEffect'], [ate], color='#A9DFBF', width=0.4)
axes[0].axhline(0, color='gray', linewidth=0.8, linestyle='--')
axes[0].set_ylabel('Estimated ATE (log wage)')
axes[0].set_title('\u00a72 Causal Effect\n(backdoor adjustment)')
axes[0].grid(axis='y', alpha=0.3)

# Panel 2: ROC curves
for label, proba, color in [('LogReg', lr_proba, '#AED6F1'),
                              ('RF',    rf_proba, '#F9E79F')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[1].plot(fpr, tpr, label=f'{label} (AUC={auc(fpr, tpr):.2f})',
                 color=color, linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].set_title('\u00a73 Supervised Learning\n(ROC curves)')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

# Panel 3: cluster scatter
sc = axes[2].scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', s=8, alpha=0.5)
axes[2].set_xlabel('PC1')
axes[2].set_ylabel('PC2')
axes[2].set_title(f'\u00a74 Unsupervised\n(K={k_chosen} worker types)')
plt.colorbar(sc, ax=axes[2], label='Cluster')

plt.suptitle('DAI Mission \u2014 Summary of Results', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('summary_figure.png', dpi=100, bbox_inches='tight')
plt.close()
print('Summary figure saved \u2192 summary_figure.png')

---
## References

*List all sources in APA format. Include data sources, key papers, and code libraries. Replace the examples below.*

Sharma, A., & Kiciman, E. (2020). DoWhy: An end-to-end library for causal inference. *arXiv:2011.04216*.

Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, *12*, 2825\u20132830.

Author, A. A., & Author, B. B. (Year). Title of article. *Journal Name*, *volume*(issue), pages. https://doi.org/...

Dataset: [Name of dataset]. Retrieved from [URL]. Accessed [date].